In [8]:
# %% [markdown]
# # Ray Tune 最终 Epoch 结算分析函数 (含 Temp 温度/EFIN/S4 全兼容版)
# 逻辑：传入 base_path，动态抓取 head_hidden_dims、alpha 以及各版本特异性 temp 温度指标。

# %%
import os
import json
import pandas as pd

# %%
def analyze_ray_results(base_path):
    """
    遍历指定的 Ray 结果路径，按模型实际超参分类对比最后 Epoch 的结算指标。
    🌟 完美补齐：自动提取 ours_s6_temp 等各种冲突温度指标，拒绝张冠李戴。
    """
    print(f"===========================================================")
    print(f" 开始解析路径: {base_path}")
    print(f"===========================================================")
    
    all_trials_data = []

    # 遍历目录
    for root, dirs, files in os.walk(base_path):
        if "params.json" in files and "result.json" in files:
            params_path = os.path.join(root, "params.json")
            result_path = os.path.join(root, "result.json")
            
            # 1. 解析参数
            try:
                with open(params_path, 'r', encoding='utf-8') as f:
                    params = json.load(f)
            except Exception:
                continue
                
            # 2. 定位最后一轮的结算指标
            last_epoch_data = None
            try:
                with open(result_path, 'r', encoding='utf-8') as f:
                    lines = [line.strip() for line in f if line.strip()]
                    if lines:
                        # 直接获取最后一行（最后 Epoch）
                        last_epoch_data = json.loads(lines[-1])
            except Exception:
                continue
                
            if last_epoch_data is not None:
                # 转换 list 为 str 方便后续做分类聚合
                head_dims = params.get("head_hidden_dims")
                head_dims_str = str(head_dims) if head_dims is not None else "None"
                
                # 提取最终结算数据 (引入全部新模型变量与核心 temp 变量兜底)
                trial_info = {
                    "trial_id": last_epoch_data.get("trial_id", os.getpid()),
                    "model": params.get("model", "None"),
                    "efin_embed_dim": params.get("efin_embed_dim", "None"), 
                    "head_hidden_dims": head_dims_str,                     
                    "alpha_wool": params.get("conflict_alpha_wool", "None"),
                    "alpha_gold": params.get("conflict_alpha_gold", "None"),
                    "alpha_walkin": params.get("conflict_alpha_walkin", "None"),
                    # 🌟 绝杀锁定：提取大盘消融矩阵中的各类温度参数 (支持 s6_temp 等)
                    "temp": params.get("ours_s6_temp", params.get("truncation_temp", params.get("align_temp", "None"))),
                    "weight_decay": params.get("weight_decay", "None"),
                    "final_epoch": last_epoch_data.get("epoch"),
                    "final_local_best_test_Y_AUUC": last_epoch_data.get("local_best_test_Target_Y_AUUC"),
                    "final_local_best_test_C_AUUC": last_epoch_data.get("local_best_test_Target_C_AUUC"),
                    "final_loss": last_epoch_data.get("loss")
                }
                all_trials_data.append(trial_info)

    # 3. 统计并打印结果
    df = pd.DataFrame(all_trials_data)

    if df.empty:
        print("❌ 没有找到任何有效的实验数据，请检查传入的路径是否正确。\n")
        return None
        
    # 🌟 关键补齐：将核心分类参数加入 temp
    core_params = ["model","head_hidden_dims", "alpha_wool", "alpha_gold", "alpha_walkin", "temp", "weight_decay"]
    for col in core_params:
        if col in df.columns:
            df[col] = df[col].fillna("None").astype(str)
        else:
            df[col] = "None"

    print(f"📊 成功加载 {len(df)} 个实验的最终结算数据。\n")
    
    # -------------------------------------------------------------
    # 维度 1：按参数组合展现各自最后 Epoch 的最高表现
    # -------------------------------------------------------------
    print("="*75)
    print(" 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC ")
    print("="*75)
    
    idx = df.groupby(core_params)["final_local_best_test_Y_AUUC"].idxmax()
    df_best_per_group = df.loc[idx].sort_values(by="final_local_best_test_Y_AUUC", ascending=False)
    
    for _, row in df_best_per_group.iterrows():
        print(f"\n[配置组合] Model: {row['model']}  | Head: {row['head_hidden_dims']}")
        print(f"            alpha_w/g/wk: {row['alpha_wool']}/{row['alpha_gold']}/{row['alpha_walkin']} | Temp: {row['temp']} | wd: {row['weight_decay']}")
        print(f"  -> 最终结算 Epoch: {row['final_epoch']} (Loss: {row['final_loss']:.4f})")
        print(f"  -> 最终 local_best_test_Y_AUUC: {row['final_local_best_test_Y_AUUC']:.6f}")
        print(f"  -> 最终 local_best_test_C_AUUC: {row['final_local_best_test_C_AUUC']:.6f}")
        print(f"  -> Trial ID: {row['trial_id']}")

    # -------------------------------------------------------------
    # 维度 2：大宽表横向对比
    # -------------------------------------------------------------
    print("\n" + "="*75)
    print(" 📋 全局最终战报总览表（按最终 local_best_test_Y_AUUC 从高到低排序） ")
    print("="*75)
    
    report_cols = core_params + ["final_epoch", "final_local_best_test_Y_AUUC", "final_local_best_test_C_AUUC"]
    print(df.sort_values(by="final_local_best_test_Y_AUUC", ascending=False)[report_cols].to_string(index=False))
    print("\n")
    
    return df

In [9]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_0717/y_pure_v10_0717_h_None_search_alpha"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_0717/y_pure_v10_0717_h_None_search_alpha
📊 成功加载 20 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET_Baseline_PureV10  | Head: None
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp: None | wd: 0.0001
  -> 最终结算 Epoch: 13 (Loss: 0.0707)
  -> 最终 local_best_test_Y_AUUC: 0.908826
  -> 最终 local_best_test_C_AUUC: 0.823297
  -> Trial ID: fb848_00007

[配置组合] Model: TARNET_Baseline_PureV10  | Head: None
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp: None | wd: 1e-05
  -> 最终结算 Epoch: 13 (Loss: 0.0706)
  -> 最终 local_best_test_Y_AUUC: 0.908396
  -> 最终 local_best_test_C_AUUC: 0.823742
  -> Trial ID: fb848_00002

[配置组合] Model: TARNET_Baseline_PureV10  | Head: None
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp: None | wd: 0.001
  -> 最终结算 Epoch: 13 (Loss: 0.0714)
  -> 最终 local_best_test_Y_AUUC: 0.908027
  -> 最终 local_best_test_C_AUUC: 0.821464
  -> Trial ID: fb848_00012

[配置组合] Model: TARNET_Baseli

In [10]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s4_conflict_0717_h_None_search_alpha"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s4_conflict_0717_h_None_search_alpha
📊 成功加载 10 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp: None | wd: 0.001
  -> 最终结算 Epoch: 13 (Loss: 0.0711)
  -> 最终 local_best_test_Y_AUUC: 0.908461
  -> 最终 local_best_test_C_AUUC: 0.823835
  -> Trial ID: d4719_00002

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 5.0/5.0/5.0 | Temp: None | wd: 0.001
  -> 最终结算 Epoch: 13 (Loss: 0.0461)
  -> 最终 local_best_test_Y_AUUC: 0.906631
  -> 最终 local_best_test_C_AUUC: 0.818672
  -> Trial ID: d4719_00001

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 1.0/1.0/1.0 | Temp: None | wd: 0.001
  -> 最终结算 Epoch: 15 (Loss: 0.0211)
  -> 最终 local_best_test_Y_AUUC: 0.901830
  -> 最终 local_best_test_C_AUUC: 0.818832
  -> Trial ID: d4719_00000

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp

In [11]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s6_conflict_0717_hNone_search_alpha_temp/y_ours_s6_conflict_0717_hNone_search_alpha_temp"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s6_conflict_0717_hNone_search_alpha_temp/y_ours_s6_conflict_0717_hNone_search_alpha_temp
📊 成功加载 48 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp: 10.0 | wd: 0.01
  -> 最终结算 Epoch: 13 (Loss: 0.0804)
  -> 最终 local_best_test_Y_AUUC: 0.909107
  -> 最终 local_best_test_C_AUUC: 0.812899
  -> Trial ID: 1d6ef_00019

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp: 20.0 | wd: 1e-05
  -> 最终结算 Epoch: 12 (Loss: 0.0722)
  -> 最终 local_best_test_Y_AUUC: 0.908289
  -> 最终 local_best_test_C_AUUC: 0.815927
  -> Trial ID: 1d6ef_00046

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp: 20.0 | wd: 0.0001
  -> 最终结算 Epoch: 12 (Loss: 0.0728)
  -> 最终 local_best_test_Y_AUUC: 0.908265
  -> 最终 local_best_test_C_AUUC: 0.816369
  -> Trial ID: 1d6ef_00034

[配置组合] Model: TARNET  | 

In [12]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10_0717/y_ours_s6_conflict_0717_hNone_search_alpha/y_ours_s6_conflict_0717_hNone_search_alpha"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10_0717/y_ours_s6_conflict_0717_hNone_search_alpha/y_ours_s6_conflict_0717_hNone_search_alpha


📊 成功加载 20 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 1.0/1.0/1.0 | Temp: 1 | wd: 1e-05
  -> 最终结算 Epoch: 18 (Loss: 0.0200)
  -> 最终 local_best_test_Y_AUUC: 0.914527
  -> 最终 local_best_test_C_AUUC: 0.828067
  -> Trial ID: fb859_00015

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp: 1 | wd: 0.0001
  -> 最终结算 Epoch: 13 (Loss: 0.0692)
  -> 最终 local_best_test_Y_AUUC: 0.910545
  -> 最终 local_best_test_C_AUUC: 0.825070
  -> Trial ID: fb859_00012

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 0.1/0.1/0.1 | Temp: 1 | wd: 1e-05
  -> 最终结算 Epoch: 38 (Loss: 0.0130)
  -> 最终 local_best_test_Y_AUUC: 0.909443
  -> 最终 local_best_test_C_AUUC: 0.839855
  -> Trial ID: fb859_00018

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp: 1 | wd: 0.001
  -> 最终结算 Epoch: 13 (Loss: 0.0704)
  -> 最终 local_best_test_Y_AUUC: 0.908925
  -> 最终 local_best_test_C_

In [14]:
import os
import json
import pandas as pd

def analyze_ray_results(base_path):
    """
    遍历指定的 Ray 结果路径，按模型实际超参分类对比最后 Epoch 的结算指标。
    🌟 升级版：全兼容 CanniUplift 的各类特异性参数 (hidden_dims, rdd_w, iptw_w, attn 等)
    """
    print(f"===========================================================")
    print(f" 开始解析路径: {base_path}")
    print(f"===========================================================")
    
    all_trials_data = []

    # 遍历目录
    for root, dirs, files in os.walk(base_path):
        if "params.json" in files and "result.json" in files:
            params_path = os.path.join(root, "params.json")
            result_path = os.path.join(root, "result.json")
            
            # 1. 解析参数
            try:
                with open(params_path, 'r', encoding='utf-8') as f:
                    params = json.load(f)
            except Exception:
                continue
                
            # 2. 定位最后一轮的结算指标
            last_epoch_data = None
            try:
                with open(result_path, 'r', encoding='utf-8') as f:
                    lines = [line.strip() for line in f if line.strip()]
                    if lines:
                        # 直接获取最后一行（最后 Epoch）
                        last_epoch_data = json.loads(lines[-1])
            except Exception:
                continue
                
            if last_epoch_data is not None:
                # 转换 list 为 str 方便后续做分类聚合
                head_dims = params.get("head_hidden_dims")
                head_dims_str = str(head_dims) if head_dims is not None else "None"
                
                # CanniUplift 使用的是 hidden_dims
                hidden_dims = params.get("hidden_dims")
                hidden_dims_str = str(hidden_dims) if hidden_dims is not None else "None"
                
                # 提取最终结算数据 (引入全部新模型变量与 CanniUplift 专属变量)
                trial_info = {
                    "trial_id": last_epoch_data.get("trial_id", os.getpid()),
                    "model": params.get("model", "None"),
                    
                    # 网络结构
                    "hidden_dims": hidden_dims_str,             # CanniUplift 用
                    "head_hidden_dims": head_dims_str,          # 其他模型用
                    "efin_embed_dim": params.get("efin_embed_dim", "None"), 
                    
                    # CanniUplift 专属超参
                    "canni_rdd_w": str(params.get("canniuplift_rdd_w", "None")),
                    "canni_redem_w": str(params.get("canniuplift_redem_w", "None")),
                    "canni_seller_w": str(params.get("canniuplift_seller_w", "None")),
                    "canni_iptw_w": str(params.get("canniuplift_iptw_w", "None")),
                    "canni_attn_dim": str(params.get("canniuplift_attn_d_dim", "None")),
                    
                    # 冲突/对齐等其他参数
                    "alpha_wool": params.get("conflict_alpha_wool", "None"),
                    "alpha_gold": params.get("conflict_alpha_gold", "None"),
                    "alpha_walkin": params.get("conflict_alpha_walkin", "None"),
                    "temp": params.get("ours_s6_temp", params.get("truncation_temp", params.get("align_temp", "None"))),
                    "weight_decay": params.get("weight_decay", "None"),
                    "lr": params.get("learning_rate", "None"),
                    
                    # 结算指标
                    "final_epoch": last_epoch_data.get("epoch"),
                    "final_local_best_test_Y_AUUC": last_epoch_data.get("local_best_test_Target_Y_AUUC"),
                    "final_local_best_test_C_AUUC": last_epoch_data.get("local_best_test_Target_C_AUUC"),
                    "final_loss": last_epoch_data.get("loss")
                }
                all_trials_data.append(trial_info)

    # 3. 统计并打印结果
    df = pd.DataFrame(all_trials_data)

    if df.empty:
        print("❌ 没有找到任何有效的实验数据，请检查传入的路径是否正确。\n")
        return None
        
    # 🌟 关键补齐：将 CanniUplift 专属变量也加入核心去重逻辑，防止参数组合被错误合并
    core_params = [
        "model", "hidden_dims", "head_hidden_dims", 
        "canni_rdd_w", "canni_redem_w", "canni_seller_w", "canni_iptw_w", "canni_attn_dim",
        "alpha_wool", "alpha_gold", "alpha_walkin", "temp", "weight_decay", "lr"
    ]
    
    for col in core_params:
        if col in df.columns:
            df[col] = df[col].fillna("None").astype(str)
        else:
            df[col] = "None"

    print(f"📊 成功加载 {len(df)} 个实验的最终结算数据。\n")
    
    # -------------------------------------------------------------
    # 维度 1：按参数组合展现各自最后 Epoch 的最高表现
    # -------------------------------------------------------------
    print("="*90)
    print(" 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC ")
    print("="*90)
    
    idx = df.groupby(core_params)["final_local_best_test_Y_AUUC"].idxmax()
    df_best_per_group = df.loc[idx].sort_values(by="final_local_best_test_Y_AUUC", ascending=False)
    
    for _, row in df_best_per_group.iterrows():
        print(f"\n[配置组合] Model: {row['model']}")
        
        # 根据模型动态打印相关超参
        if row['model'] == "CanniUplift":
            print(f"            Structure: hidden_dims={row['hidden_dims']} | lr={row['lr']} | wd={row['weight_decay']}")
            print(f"            Canni_Weights: rdd={row['canni_rdd_w']}, redem={row['canni_redem_w']}, seller={row['canni_seller_w']}, iptw={row['canni_iptw_w']}")
            print(f"            Canni_Attn: dim={row['canni_attn_dim']}")
        else:
            print(f"            Structure: head_dims={row['head_hidden_dims']} | lr={row['lr']} | wd={row['weight_decay']}")
            print(f"            Multi-Task: alpha_w/g/wk: {row['alpha_wool']}/{row['alpha_gold']}/{row['alpha_walkin']} | Temp: {row['temp']}")
            
        print(f"  -> 最终结算 Epoch: {row['final_epoch']} (Loss: {row['final_loss']:.4f})")
        print(f"  -> 最终 local_best_test_Y_AUUC: {row['final_local_best_test_Y_AUUC']:.6f}")
        print(f"  -> 最终 local_best_test_C_AUUC: {row['final_local_best_test_C_AUUC']:.6f}")
        print(f"  -> Trial ID: {row['trial_id']}")

    # -------------------------------------------------------------
    # 维度 2：大宽表横向对比
    # -------------------------------------------------------------
    print("\n" + "="*90)
    print(" 📋 全局最终战报总览表（按最终 local_best_test_Y_AUUC 从高到低排序） ")
    print("="*90)
    
    # 为了宽表显示紧凑，可以剔除一些全是 "None" 的无效列
    report_cols = core_params + ["final_epoch", "final_local_best_test_Y_AUUC", "final_local_best_test_C_AUUC"]
    df_display = df.sort_values(by="final_local_best_test_Y_AUUC", ascending=False)[report_cols]
    
    # 自动过滤掉大宽表中所有值都是 "None" 的列，保持表格清爽
    cols_to_show = [col for col in df_display.columns if not (df_display[col] == "None").all()]
    print(df_display[cols_to_show].to_string(index=False))
    print("\n")
    
    return df

In [15]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/CanniUplift/y_baseline_canniuplift_paper_grid/y_baseline_canniuplift_paper_grid"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/CanniUplift/y_baseline_canniuplift_paper_grid/y_baseline_canniuplift_paper_grid
📊 成功加载 61 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: CanniUplift
            Structure: hidden_dims=[128, 64, 32] | lr=0.001 | wd=0.0001
            Canni_Weights: rdd=0.5, redem=0.5, seller=1.0, iptw=1.0
            Canni_Attn: dim=16
  -> 最终结算 Epoch: 11 (Loss: nan)
  -> 最终 local_best_test_Y_AUUC: 0.763157
  -> 最终 local_best_test_C_AUUC: 0.840742
  -> Trial ID: 54163_00007

[配置组合] Model: CanniUplift
            Structure: hidden_dims=[128, 64, 32] | lr=0.001 | wd=0.001
            Canni_Weights: rdd=0.5, redem=0.5, seller=1.0, iptw=1.0
            Canni_Attn: dim=16
  -> 最终结算 Epoch: 11 (Loss: nan)
  -> 最终 local_best_test_Y_AUUC: 0.763157
  -> 最终 local_best_test_C_AUUC: 0.840742
  -> Trial ID: 54163_00037

[配置组合] Model: CanniUplift
            Structure: hidden_dims=[128, 64, 32] | lr=0.001 | wd=0.01
            